In [ ]:
import importlib
import pandas as pd
import os
import sys
from pathlib import Path

# Charger et recharger le module simplifie
import Fcn_GetDatasetFromISMN as fct_gee
importlib.reload(fct_gee)

# Importer les fonctions nécessaires
from Fcn_GetDatasetFromISMN import (
    get_assets_before_and_after_filters,
    create_bbox_around_point,
    download_site_datasets,
    batch_download_ismn_sites,
    BASE_DEFAULT,
    get_field_bbox_from_landcover,
    configure_inputs,
    discover_collections,
    build_compact_output_stem
)

## 1. Imports et configuration

# Multi-Site ISMN Dataset Download from OpenLandMap

This notebook downloads SoilGrids datasets (clay, silt, sand) for multiple ISMN sites.

**Workflow:**
1. Load ISMN site coordinates from your HRSM_Grandvillers analysis
2. Create bounding boxes around each site
3. Download datasets for each site individually
4. Generate metadata and logs for tracking

## 2. Load ISMN Sites

Load the ISMN site coordinates. This should be the same dataframe used in your HRSM_Grandvillers notebook.

In [42]:
# ============================================================================
# Load your ISMN sites data
# ============================================================================

# Option 1: If you have the kept_stations_summary.csv from HRSM_Grandvillers
try:
    ismn_sites = pd.read_csv('ISMN_PSA_merged_sites_with_features.csv')
    print(f"✓ Loaded {len(ismn_sites)} sites from kept_stations_summary.csv")
except FileNotFoundError:
    print("kept_stations_summary.csv not found")
    ismn_sites = None

# Option 2: Alternative path (if CSV is elsewhere)
if ismn_sites is None:
    try:
        ismn_sites = pd.read_csv('./ISMN_sites.csv')
        print(f"✓ Loaded {len(ismn_sites)} sites from ISMN_sites.csv")
    except FileNotFoundError:
        print("ISMN_sites.csv not found")
        ismn_sites = None

# Check the loaded data
if ismn_sites is not None:
    print(f"\nSites shape: {ismn_sites.shape}")
    print(f"Columns: {list(ismn_sites.columns)}")
    print(f"\nFirst few rows:")
    display(ismn_sites.head())
else:
    print("\n⚠️  Please specify the path to your ISMN sites CSV file")

✓ Loaded 72 sites from kept_stations_summary.csv

Sites shape: (72, 9)
Columns: ['ID', 'Longitude', 'Latitude', 'network', 'station_folder', 'surface_depth', 'root_depth', 'surface_path', 'root_path']

First few rows:


,ID,Longitude,Latitude,network,station_folder,surface_depth,root_depth,surface_path,root_path
0,0,13.51000,52.40145,Berlin,PSA17suedpark,0.05,0.45,/content/gdrive/MyDrive/NIFA_Download/StartFol...,/content/gdrive/MyDrive/NIFA_Download/StartFol...
1,1,0.99307,52.09465,COSMOS-UK,Elmsett,0.05,0.50,/content/gdrive/MyDrive/NIFA_Download/StartFol...,/content/gdrive/MyDrive/NIFA_Download/StartFol...
2,2,0.79592,52.33639,COSMOS-UK,Euston,0.05,0.50,/content/gdrive/MyDrive/NIFA_Download/StartFol...,/content/gdrive/MyDrive/NIFA_Download/StartFol...
3,3,0.51073,52.61777,COSMOS-UK,Fincham,0.05,0.50,/content/gdrive/MyDrive/NIFA_Download/StartFol...,/content/gdrive/MyDrive/NIFA_Download/StartFol...
4,4,0.32018,51.22856,COSMOS-UK,Hadlow,0.10,0.50,/content/gdrive/MyDrive/NIFA_Download/StartFol...,/content/gdrive/MyDrive/NIFA_Download/StartFol...


## 3. Paramètres de téléchargement

Configurez les paramètres pour le téléchargement batch.

In [43]:
# ============================================================================
# Configuration des paramètres de téléchargement - VERSION SIMPLIFIEE
# ============================================================================

# Paramètres de téléchargement
SOIL_TYPES = ["bulk","silt","clay","sand","Ksat","dem"]
STAT = "m"
RESOLUTION = "30m"

# Une seule logique: on cible les champs détectés via Landcover
# Ajouter 'dem' permet aussi de générer automatiquement slope/aspect/twi
SEARCH_RADIUS_KM = 5      # Rayon d'analyse autour du site
FIELD_BUFFER_KM = 2       # Taille du bbox final autour du site si cropland détecté
REQUIRE_CROPLAND = True   # Ne garder que les sites dans des champs

BASE_URL = BASE_DEFAULT

# Output directories
OUTPUT_BASE_DIR = "./gee_export/sites"
METADATA_OUTPUT = "./gee_export/ISMN_metadata.csv"
LOG_OUTPUT = "./gee_export/download_log.csv"

print("Paramètres de téléchargement:")
print(f"  Soil types: {SOIL_TYPES}")
print(f"  Stat: {STAT}")
print(f"  Resolution: {RESOLUTION}")
print("  Method: Landcover cropland only")
print(f"  Search radius: {SEARCH_RADIUS_KM} km")
print(f"  Field buffer: {FIELD_BUFFER_KM} km")
print(f"  Require cropland: {REQUIRE_CROPLAND}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  Metadata: {METADATA_OUTPUT}")
print(f"  Log: {LOG_OUTPUT}")

Paramètres de téléchargement:
  Soil types: ['bulk', 'silt', 'clay', 'sand', 'Ksat', 'dem']
  Stat: m
  Resolution: 30m
  Method: Landcover cropland only
  Search radius: 5 km
  Field buffer: 2 km
  Require cropland: True
  Output base: ./gee_export/sites
  Metadata: ./gee_export/ISMN_metadata.csv
  Log: ./gee_export/download_log.csv


## 4. Preview: Test avec 1 site

Avant de lancer le téléchargement complet, testez le premier site pour vérifier s'il est bien détecté comme cropland.

In [44]:
# Test avec le premier site
if ismn_sites is not None and len(ismn_sites) > 0:
    test_site = ismn_sites.iloc[0]
    test_site_id = test_site.get("ID") or "site_0"
    test_lon = test_site.get("Longitude")
    test_lat = test_site.get("Latitude")
    
    print(f"Test site: {test_site_id}")
    print(f"  Coordinates: ({test_lat:.4f}, {test_lon:.4f})")
    
    field_info = get_field_bbox_from_landcover(
        test_lon,
        test_lat,
        field_buffer_km=FIELD_BUFFER_KM,
        search_radius_km=SEARCH_RADIUS_KM,
        verbose=True,
    )
    
    if field_info is None:
        print("  Site not classified as cropland in the search window")
    else:
        print(f"  Cropland confidence: {field_info['confidence']}%")
        print(f"  Field bbox: {[round(x, 4) for x in field_info['bbox']]}")
        print("\n✓ Field-based parameters look good. Ready for batch download!")
else:
    print("⚠️  No ISMN sites loaded. Please configure the data source in the previous cells.")

Test site: site_0
  Coordinates: (52.4014, 13.5100)
  Cropland detected: 10.1% of sampled pixels in a 5 km search window
  Field bbox buffer: 2 km
  Cropland confidence: 10.1%
  Field bbox: [np.float64(13.492), np.float64(52.3834), np.float64(13.528), np.float64(52.4195)]

✓ Field-based parameters look good. Ready for batch download!


In [45]:
params = configure_inputs(
    soil_type=["dem"],
    stat=STAT,
    resolution=RESOLUTION,
    box_zone=field_info['bbox'],
    base=BASE_URL,
    strict=False,
)

## 5. Batch Download - All Sites

⚠️ **This may take a while** depending on:
- Number of sites (72)
- Network speed
- Server response time
- Earth Engine landcover checks

Only sites classified as **cropland** in WorldCover are kept. Each site will have its own subdirectory in `gee_export/sites/`.

In [47]:
if ismn_sites is not None and len(ismn_sites) > 0:
    batch_result = batch_download_ismn_sites(
        ismn_sites_df=ismn_sites,
        output_base_dir=OUTPUT_BASE_DIR,
        soil_types=SOIL_TYPES,
        stat=STAT,
        resolution=RESOLUTION,
        field_buffer_km=FIELD_BUFFER_KM,
        search_radius_km=SEARCH_RADIUS_KM,
        base=BASE_URL,
        require_cropland=REQUIRE_CROPLAND,
        verbose=True,
        metadata_output_path=METADATA_OUTPUT,
        log_output_path=LOG_OUTPUT,
    )

    results = batch_result['results']
else:
    print("⚠️  No ISMN sites available. Please load your data first.")


######################################################################
# BATCH DOWNLOAD - 72 sites ISMN
# Output dir: ./gee_export/sites
# Field buffer: 2km, Search radius: 5km, Stat: m, Resolution: 30m
######################################################################

[1/72] Traitement site_0...
  Cropland detected: 10.1% of sampled pixels in a 5 km search window
  Field bbox buffer: 2 km

SITE: site_0
  Position: (52.4014, 13.5100)
  Landcover: Cropland
  Confidence: 10.1%
  bbox: [13.492, 52.3834, 13.528, 52.4195]
Collections trouvées: 8
COLLECTION : Long-term soil bulk density for fine earth
ITEMS      : 1
COLLECTION : MERIT geomorphometry
ITEMS      : 1
COLLECTION : OpenLandMap long-term sand content
ITEMS      : 1
COLLECTION : Copernius DEM digital surface model
ITEMS      : 1
COLLECTION : OpenLandMap-soildb: Bulk density ﬁne earth [kg/m3]
ITEMS      : 1
COLLECTION : OpenLandMap-soildb: Soil texture fraction sand [%]
ITEMS      : 1
COLLECTION : OpenLandMap-soildb: Soil text

## 6. Analyse des résultats

In [48]:
# Charger et analyser le log
if os.path.exists(LOG_OUTPUT):
    log_df = pd.read_csv(LOG_OUTPUT)
    print(f"\n{'='*70}")
    print(f"RÉSULTATS DU TÉLÉCHARGEMENT")
    print(f"{'='*70}")
    print(f"Total sites: {len(log_df)}")
    print(f"✓ Succès: {(log_df['status'] == 'success').sum()}")
    print(f"❌ Échecs: {(log_df['status'] == 'failed').sum()}")
    print(f"{'='*70}\n")
    
    print(f"Statistiques des fichiers téléchargés:")
    success_stats = log_df[log_df['status'] == 'success']['num_files'].describe()
    print(success_stats)
    
    print(f"\nSites avec erreurs:")
    failed = log_df[log_df['status'] == 'failed']
    if len(failed) > 0:
        display(failed[['site_id', 'error']])
    else:
        print("✓ Tous les sites ont été téléchargés avec succès!")
else:
    print("⚠️  Log file not found")


RÉSULTATS DU TÉLÉCHARGEMENT
Total sites: 72
✓ Succès: 72
❌ Échecs: 0

Statistiques des fichiers téléchargés:
count    72.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max      20.0
Name: num_files, dtype: float64

Sites avec erreurs:
✓ Tous les sites ont été téléchargés avec succès!


## 7. Structure de sortie

Affiche l'organisation des fichiers téléchargés.

In [49]:
import os
from pathlib import Path

if os.path.exists(OUTPUT_BASE_DIR):
    print(f"Structure de gee_export/sites/:\n")
    print(f"{OUTPUT_BASE_DIR}/")
    
    sites = [d for d in os.listdir(OUTPUT_BASE_DIR) if os.path.isdir(os.path.join(OUTPUT_BASE_DIR, d))]
    sites_sorted = sorted(sites)[:10]  # Afficher les 10 premiers
    
    for site in sites_sorted:
        site_path = os.path.join(OUTPUT_BASE_DIR, site)
        files = os.listdir(site_path)
        tif_files = [f for f in files if f.endswith('.tif')]
        print(f"  └─ {site}/")
        for i, tif in enumerate(tif_files[:3]):  # Show first 3 files
            size_mb = os.path.getsize(os.path.join(site_path, tif)) / (1024*1024)
            symbol = "├─" if i < 2 or len(tif_files) <= 3 else "└─"
            print(f"      {symbol} {tif} ({size_mb:.2f} MB)")
        if len(tif_files) > 3:
            print(f"      └─ ... ({len(tif_files)-3} more files)")
    
    if len(sites) > 10:
        print(f"  ... ({len(sites)-10} more sites)")
    
    print(f"\n✓ Total sites with data: {len(sites)}")
else:
    print(f"⚠️  Output directory not found: {OUTPUT_BASE_DIR}")

Structure de gee_export/sites/:

./gee_export/sites/
  └─ 1/
      ├─ ksat_m_1km_100cm.tif (0.00 MB)
      ├─ bulk_m_30m_0cm_30cm.tif (0.02 MB)
      └─ dem_m_30m_depth_aspect.tif (0.07 MB)
      └─ ... (17 more files)
  └─ 10/
      ├─ ksat_m_1km_100cm.tif (0.00 MB)
      ├─ bulk_m_30m_0cm_30cm.tif (0.02 MB)
      └─ dem_m_30m_depth_aspect.tif (0.07 MB)
      └─ ... (17 more files)
  └─ 11/
      ├─ ksat_m_1km_100cm.tif (0.00 MB)
      ├─ bulk_m_30m_0cm_30cm.tif (0.02 MB)
      └─ dem_m_30m_depth_aspect.tif (0.07 MB)
      └─ ... (17 more files)
  └─ 12/
      ├─ ksat_m_1km_100cm.tif (0.00 MB)
      ├─ bulk_m_30m_0cm_30cm.tif (0.02 MB)
      └─ dem_m_30m_depth_aspect.tif (0.07 MB)
      └─ ... (17 more files)
  └─ 13/
      ├─ ksat_m_1km_100cm.tif (0.00 MB)
      ├─ bulk_m_30m_0cm_30cm.tif (0.03 MB)
      └─ dem_m_30m_depth_aspect.tif (0.07 MB)
      └─ ... (17 more files)
  └─ 14/
      ├─ ksat_m_1km_100cm.tif (0.00 MB)
      ├─ bulk_m_30m_0cm_30cm.tif (0.02 MB)
      └─ dem_m_30m_de